# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# Clone your specific repository
!git clone https://github.com/PrathamDudani/FlyRank_Assignment.git

# Change into the repository's folder
%cd FlyRank_Assignment

Cloning into 'FlyRank_Assignment'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 137 (delta 42), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.95 MiB | 8.55 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/FlyRank_Assignment/FlyRank_Assignment


In [15]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
valid = df[df["avg_position"] > 0].copy()   # avg_position=0 means no data — must filter

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [16]:
REASON_CODES = ["CTR_BELOW_POSITION_EXPECTED", "NO_FLAG"]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
# ── SECTION 2: Build the ranked queue ────────────────────────

# signal 1 bucket table (position vs ctr) — using your own bins, not the pre-built tier
valid["position_bucket"] = pd.cut(
    valid["avg_position"], bins=[0,3,6,10,20,1000],
    labels=["1-3","4-6","7-10","11-20","20+"]
)
signal1_table = valid.groupby("position_bucket")["ctr"].agg(["mean","median","count"])
print(signal1_table)

# signal 2 bucket table (staleness vs engagement)
valid["staleness_bucket"] = pd.cut(
    valid["days_since_last_update"], bins=[0,90,180,365,10000],
    labels=["<90d","90-180d","180-365d","365d+"]
)
signal2_table = valid.groupby("staleness_bucket")["engagement_rate"].agg(["mean","median","count"])
print(signal2_table)

# --- read the two tables above, THEN set these from what they actually show ---
expected_ctr = valid.groupby("position_bucket")["ctr"].transform("mean")
ctr_gap = expected_ctr - valid["ctr"]

GAP_THRESHOLD = None          # e.g. set to something like the median gap or a spread you saw
IMPRESSION_THRESHOLD = None   # e.g. set from valid["impressions_90d"].describe()

underperforming = (ctr_gap > GAP_THRESHOLD).astype(int)
visible = (valid["impressions_90d"] >= IMPRESSION_THRESHOLD).astype(int)

valid["score"] = underperforming * visible * valid["impressions_90d"]
valid["reason_code"] = np.where(
    (underperforming == 1) & (visible == 1),
    "CTR_BELOW_POSITION_EXPECTED", "NO_FLAG"
)
valid["action"] = np.where(valid["score"] > 0, "REVIEW_FOR_REFRESH", "NO_ACTION")

ranked = valid.sort_values("score", ascending=False)
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

                     mean  median  count
position_bucket                         
1-3              2.714303    0.00   1141
4-6              0.931510    0.21   4801
7-10             0.459807    0.12   7041
11-20            0.323443    0.10   7273
20+              0.211333    0.00   8539
                      mean  median  count
staleness_bucket                         
<90d              2.686162     0.0  19475
90-180d           2.408497     0.0   9162
180-365d          2.306732     0.0    153
365d+             0.000000     0.0      5


/tmp/ipykernel_2655/4242733187.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1_table = valid.groupby("position_bucket")["ctr"].agg(["mean","median","count"])
/tmp/ipykernel_2655/4242733187.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2_table = valid.groupby("staleness_bucket")["engagement_rate"].agg(["mean","median","count"])
/tmp/ipykernel_2655/4242733187.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence th

In [19]:
expected_ctr = valid.groupby("position_bucket")["ctr"].transform("mean")
ctr_gap = expected_ctr - valid["ctr"]

print(ctr_gap.describe())
print(valid["impressions_90d"].describe())

GAP_THRESHOLD = ctr_gap.quantile(0.75)
IMPRESSION_THRESHOLD = valid["impressions_90d"].quantile(0.50)

underperforming = (ctr_gap > GAP_THRESHOLD).astype(int)
visible = (valid["impressions_90d"] >= IMPRESSION_THRESHOLD).astype(int)

valid["score"] = underperforming * visible * valid["impressions_90d"]
valid["reason_code"] = np.where(
    (underperforming == 1) & (visible == 1),
    "CTR_BELOW_POSITION_EXPECTED", "NO_FLAG"
)
valid["action"] = np.where(valid["score"] > 0, "REVIEW_FOR_REFRESH", "NO_ACTION")

ranked = valid.sort_values("score", ascending=False)
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

/tmp/ipykernel_2655/4256293525.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  expected_ctr = valid.groupby("position_bucket")["ctr"].transform("mean")


count    2.879500e+04
mean     6.415736e-18
std      3.192461e+00
min     -9.978867e+01
25%      9.133271e-02
50%      2.113327e-01
75%      4.415101e-01
max      2.714303e+00
Name: ctr, dtype: float64
count     28795.000000
mean       5417.909984
std       17152.423172
min           1.000000
25%         118.000000
50%         828.000000
75%        3884.500000
max      517715.000000
Name: impressions_90d, dtype: float64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
top20 = ranked.head(20)
print(top20[["content_id","action","reason_code","score"]])

                 content_id              action                  reason_code  \
6653   content_5fe46e04994d  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
17812  content_aaef01a50def  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
26844  content_8c19996aa890  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
21819  content_4c36c775b818  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
29879  content_1a9e894be2e2  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
18870  content_db5989a78dd3  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
14090  content_44e481c8f55b  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
26531  content_cb112fce36be  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
21565  content_9532f197bbc8  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
7678   content_8451fc6f034d  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
27478  content_008fb02c46cb  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
22028  content_73c54f78c06a  REVIEW_FOR_

In [21]:
top20 = ranked.head(20)
print(top20[["content_id","action","reason_code","score","avg_position","ctr","impressions_90d"]])

                 content_id              action                  reason_code  \
6653   content_5fe46e04994d  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
17812  content_aaef01a50def  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
26844  content_8c19996aa890  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
21819  content_4c36c775b818  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
29879  content_1a9e894be2e2  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
18870  content_db5989a78dd3  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
14090  content_44e481c8f55b  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
26531  content_cb112fce36be  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
21565  content_9532f197bbc8  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
7678   content_8451fc6f034d  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
27478  content_008fb02c46cb  REVIEW_FOR_REFRESH  CTR_BELOW_POSITION_EXPECTED   
22028  content_73c54f78c06a  REVIEW_FOR_

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [22]:
leak_cols = ["trend_direction", "trend_pct", "is_declining_label"]
used_cols = ["ctr", "avg_position", "impressions", "engagement_rate", "days_since_update"]
assert not set(leak_cols) & set(used_cols)
print("No leakage columns used.")

No leakage columns used.


In [23]:
leak_cols = ["trend_direction", "trend_pct"]
used_cols = ["ctr", "avg_position", "impressions_90d", "engagement_rate", "days_since_last_update"]
assert not set(leak_cols) & set(used_cols)
print("No leakage columns used:", leak_cols)

No leakage columns used: ['trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.